# Detector systematics summary (WireMod + DENT)

Load Product B outputs from `wiremod.ipynb` and `dent.ipynb`, overlay fractional
uncertainties (WireMod YZ, WireMod XTXW, DENT, and quadrature total), and write
a single `Detector/detector_syst_dict.npz` for `systematics-summary.ipynb`.

Comparison style follows `scripts/plot_dent_old_vs_new_vs_wiremod.py`.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path("/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana")
sys.path.insert(0, str(REPO))

from analysis_village.numucc_1p0pi.dataset_locations import PLOTS_BASE, default_syst_disk_root
from analysis_village.numucc_1p0pi.final_selected_evt_vars import (
    CORE_SELECTED_EVT_VARIABLE_CONFIGS,
    with_final_selected_evt_variables,
)
from analysis_village.numucc_1p0pi.syst_disk_layout import FILE_DETECTOR, SUB_DETECTOR
from analysis_village.numucc_1p0pi.syst_detvar_common import (
    combine_wiremod_dent_detector_dict,
    load_detector_dict_npz,
    log,
    plot_detector_source_frac_unc,
    save_detector_npz,
)
from analysis_village.numucc_1p0pi.utils import dpi
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig


In [ ]:
PLOTS = Path(os.environ.get("NUMUCC_PLOTS_BASE", PLOTS_BASE))
WIREMOD_NPZ = Path(os.environ.get(
    "WIREMOD_DETECTOR_NPZ",
    str(PLOTS / "systematics-final-archive" / "WireMod" / SUB_DETECTOR / FILE_DETECTOR),
))
DENT_NPZ = Path(os.environ.get(
    "DENT_DETECTOR_NPZ",
    str(PLOTS / "systematics-final-archive" / "DENT" / SUB_DETECTOR / FILE_DETECTOR),
))

# Combined Detector lands on the PRL Product B syst disk by default.
SYST_DISK_ROOT = Path(os.environ.get(
    "NUMUCC_SYST_DISK_ROOT",
    str(default_syst_disk_root()),
))
OUT_DIR = SYST_DISK_ROOT / SUB_DETECTOR
FIG_DIR = SYST_DISK_ROOT / "plots" / "Detector"
OUT_NPZ = OUT_DIR / FILE_DETECTOR
SAVE_FIGS = True

for d in (OUT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("WireMod:", WIREMOD_NPZ, "exists=", WIREMOD_NPZ.is_file())
print("DENT:", DENT_NPZ, "exists=", DENT_NPZ.is_file())
print("OUT:", OUT_NPZ)


In [ ]:
wiremod_dict = load_detector_dict_npz(WIREMOD_NPZ)
dent_dict = load_detector_dict_npz(DENT_NPZ)
detector_dict = combine_wiremod_dent_detector_dict(wiremod_dict, dent_dict)

save_detector_npz(
    detector_dict,
    OUT_NPZ,
    manifest={
        "sources": ["WireModYZ", "WireModXTXW", "DENT"],
        "wiremod_npz": str(WIREMOD_NPZ),
        "dent_npz": str(DENT_NPZ),
        "n_vars": len(detector_dict.get("detector", {})),
    },
)
print("combined keys:", sorted(k for k in detector_dict if k.startswith("detector")))
print("n_vars:", len(detector_dict.get("detector", {})))


## Overlay plots — WireMod YZ / XTXW / DENT / total


In [ ]:
var_configs = with_final_selected_evt_variables(list(CORE_SELECTED_EVT_VARIABLE_CONFIGS))
if not any(vc.var_save_name == "integrated" for vc in var_configs):
    var_configs = [VariableConfig.all_events()] + list(var_configs)

for vc in var_configs:
    vsn = vc.var_save_name
    if vsn not in detector_dict.get("detector", {}):
        continue
    save_path = FIG_DIR / f"detector_frac_unc__{vsn}.png" if SAVE_FIGS else None
    plot_detector_source_frac_unc(
        detector_dict,
        vc,
        source_order=("wiremod_yz", "wiremod_xtxw", "DENT", "Detector"),
        save_path=save_path,
        dpi=dpi,
    )


## Done

`systematics-summary.ipynb` loads this file from `Detector/detector_syst_dict.npz`
(or via `SOURCE_DISK_ROOTS["Detector"]`).
